# Baseline
Group: ***'NLP (No Language Proficiency)'***

For baseline we use:
- pretrained BERT model -> *model_name =  "google-bert/bert-base-cased"*
- tuning on -> EWT text *en_ewt-ud-train* recognize specific labels (locations and persons)

Other models on [Hugging Face](https://huggingface.co/transformers/v3.3.1/pretrained_models.html)

Main idea: pretrained model knows relationships between words, but not the labels like Paris is a location, that's why we specify that in tuning dataset

In [9]:
!pip install datasets transformers evaluate torch seqeval

In [32]:
!pip3 install torch --index-url https://download.pytorch.org/whl/cu126

Looking in indexes: https://download.pytorch.org/whl/cu126


In [1]:
# Libraries and imports
from datasets import Dataset
from transformers import (AutoTokenizer, AutoModelForTokenClassification, DataCollatorForTokenClassification, AutoConfig, set_seed)
from torch.utils.data import DataLoader
import torch
import random
import evaluate
from tqdm.auto import tqdm
import numpy as np


### Loading tuning dataset

First step is parsing a dataset, or in other words readint he raw data file and converting it into lists and dictionaries that python can use. 

Note: in the training file there is stephen in the last column, it's just an annotators name

In [1]:
# Function to parse the datasets 

def parse_iob2_file(filepath):
    sentences = []
    labels = []
    tokens = []
    tags = []
    with open(filepath, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#"):
                if tokens:
                    sentences.append(tokens)
                    labels.append(tags)
                    tokens = []
                    tags = []
                continue
            splits = line.split("\t")
            tokens.append(splits[1])
            tags.append(splits[2])

    return sentences, labels


In [7]:
# Testing function

dev_sentences, dev_labels = parse_iob2_file("en_ewt-ud-dev.iob2")
dev_sentences[:1], dev_labels[:1]

([['where',
   'can',
   'I',
   'get',
   'morcillas',
   'in',
   'tampa',
   'bay',
   ',',
   'I',
   'will',
   'like',
   'the',
   'argentinian',
   'type',
   ',',
   'but',
   'I',
   'will',
   'to',
   'try',
   'anothers',
   'please',
   '?']],
 [['O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'B-LOC',
   'I-LOC',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O']])

In [12]:
# Parsing the rest of the datasets needed for training and evaluation

train_sentences, train_labels = parse_iob2_file("en_ewt-ud-train.iob2")
test_sentences, test_labels = parse_iob2_file("en_ewt-ud-test-masked.iob2")

In [15]:
# Making dataset compatible with Hugging Face

train_dataset = Dataset.from_dict({"tokens": train_sentences, "ner_tags": train_labels})
dev_dataset = Dataset.from_dict({"tokens": dev_sentences, "ner_tags": dev_labels})
test_dataset = Dataset.from_dict({"tokens": test_sentences, "ner_tags": test_labels})

In [16]:
# For inspection

print(train_dataset[0])

{'tokens': ['Where', 'in', 'the', 'world', 'is', 'Iguazu', '?'], 'ner_tags': ['O', 'O', 'O', 'O', 'O', 'B-LOC', 'O']}


In [18]:
# Getting unique labels

all_labels = set(l for seq in train_labels + dev_labels + test_labels for l in seq)
label_list = sorted(all_labels)
print(label_list)

['B-LOC', 'B-ORG', 'B-PER', 'I-LOC', 'I-ORG', 'I-PER', 'O']


In [19]:
# Formating labels for pyTorch (having two dics for faster and safer coding)

label2id = {l: i for i, l in enumerate(label_list)} # Converting labels to IDs, since pyTorch models work with integers
id2label = {i: l for l, i in label2id.items()} # Reverse mapping for ID to label, for evalutation later

label2id, id2label

({'B-LOC': 0,
  'B-ORG': 1,
  'B-PER': 2,
  'I-LOC': 3,
  'I-ORG': 4,
  'I-PER': 5,
  'O': 6},
 {0: 'B-LOC',
  1: 'B-ORG',
  2: 'B-PER',
  3: 'I-LOC',
  4: 'I-ORG',
  5: 'I-PER',
  6: 'O'})

In [20]:
# Label to id conversion function

def labels_to_ids(labels, label2id):
    L = []
    for seq in labels:
        seq_ids = []
        for tag in seq:
            if tag in label2id:
                seq_ids.append(label2id[tag])
            else:
                raise ValueError(f"{tag} not found")
        L.append(seq_ids)

    return L

In [21]:
# Convert string labels to IDs

train_label_ids = labels_to_ids(train_labels, label2id)
dev_label_ids = labels_to_ids(dev_labels, label2id)
test_label_ids = labels_to_ids(test_labels, label2id)

In [27]:
# Inspection

print(train_label_ids[0])
print(f'Label 6: {id2label[6]}')
print(f'Label 0: {id2label[0]}')

[6, 6, 6, 6, 6, 0, 6]
Label 6: O
Label 0: B-LOC


In [25]:
# Updating datasets with integer labels, so it works with Hugging Face model

train_dataset = Dataset.from_dict({"tokens": train_sentences, "ner_tags": train_label_ids})
dev_dataset = Dataset.from_dict({"tokens": dev_sentences, "ner_tags": dev_label_ids})
test_dataset = Dataset.from_dict({"tokens": test_sentences, "ner_tags": test_label_ids})

In [28]:
# Inspection

print(train_dataset[0])

{'tokens': ['Where', 'in', 'the', 'world', 'is', 'Iguazu', '?'], 'ner_tags': [6, 6, 6, 6, 6, 0, 6]}


**Datasets preprocessing part all together:**

In [2]:
# Needed functions

# Dataset parsing function
def parse_iob2_file(filepath):
    sentences = []
    labels = []
    tokens = []
    tags = []
    with open(filepath, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#"):
                if tokens:
                    sentences.append(tokens)
                    labels.append(tags)
                    tokens = []
                    tags = []
                continue
            splits = line.split("\t")
            tokens.append(splits[1])
            tags.append(splits[2])

    return sentences, labels

# Label to id conversion function
def labels_to_ids(labels, label2id):
    L = []
    for seq in labels:
        seq_ids = []
        for tag in seq:
            if tag in label2id:
                seq_ids.append(label2id[tag])
            else:
                raise ValueError(f"{tag} not found")
        L.append(seq_ids)

    return L

In [3]:
# Parsing the datasets
dev_sentences, dev_labels = parse_iob2_file("en_ewt-ud-dev.iob2")
train_sentences, train_labels = parse_iob2_file("en_ewt-ud-train.iob2")
test_sentences, test_labels = parse_iob2_file("en_ewt-ud-test-masked.iob2")

# Build label list and label2id/id2label mappings
all_labels = set(l for seq in train_labels + dev_labels for l in seq) # Collecting all unique labels
label_list = sorted(all_labels)
label2id = {l: i for i, l in enumerate(label_list)} # Converting labels to IDs, since pyTorch models work with integers
id2label = {i: l for l, i in label2id.items()} # Reverse mapping for ID to label, for evalutation later

train_label_ids = labels_to_ids(train_labels, label2id)
dev_label_ids = labels_to_ids(dev_labels, label2id)
test_label_ids = labels_to_ids(test_labels, label2id)

# Making dataset compatible with Hugging Face
train_dataset = Dataset.from_dict({"tokens": train_sentences, "ner_tags": train_label_ids})
dev_dataset = Dataset.from_dict({"tokens": dev_sentences, "ner_tags": dev_label_ids})
test_dataset = Dataset.from_dict({"tokens": test_sentences, "ner_tags": test_label_ids})


In [4]:
# Final inspection

print(train_dataset[0])

{'tokens': ['Where', 'in', 'the', 'world', 'is', 'Iguazu', '?'], 'ner_tags': [6, 6, 6, 6, 6, 0, 6]}


### Tokenizer

In this part we are loading the tokenizer that matches the pretrained BERT model (so it splits and encodes text the same way BERT expects). Here we do not load the BERT model itself just the preprocessing tool.

The loading of pretrained model is: 

```python
model = AutoModelForTokenClassification.from_pretrained(model_name, config=config)
```

In [5]:
# Our selected pretrained model 
model_name = "google-bert/bert-base-cased"

# Define hyperparameters (now set to assignment's parameters)
learning_rate = 2e-5
num_train_epochs = 3

# Set random seeds
set_seed(42)

In [6]:
# Loading tokenizer and model configuration

tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True) # tool that prepares text for BERT, using faster implementation
config = AutoConfig.from_pretrained(model_name, num_labels=len(label_list), id2label=id2label, label2id=label2id) # loads configuration for pretrained model, setting how many NER labels there are, provided conext with rest

In [7]:
# Function to tokenize dataset into subwords and align labels with those

def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"], # Lists of tokens for each sentence, e.g. [["EU", "rejects", "Germany", "call", "to", "boycott", "British", "lamb", "."]]
        max_length=128, #  Limits the total number of tokens (including special tokens) to 128. Longer sequences are truncated.
        padding=False, # All sentences keeps their original length, not making all sentences the same length
        truncation=True, # If sequence is longer than max_length, it will be truncated to fit the model's input size.
        is_split_into_words=True # Indicates that the input is already split into words
    )
    
    all_labels = []
    for i, labels in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        label_ids = []
        prev_word_id = None
        for word_id in word_ids:
            if word_id is None:
                label_ids.append(-100)
            elif word_id == prev_word_id:
                label_ids.append(-100)
            else:
                label_ids.append(labels[word_id])
            prev_word_id = word_id
        all_labels.append(label_ids)
    tokenized_inputs["labels"] = all_labels
    return tokenized_inputs


`tokenize_and_align_labels(examples)`

**Example:**

- `tokens = ["John", "Smith"]`
- `labels = [1, 2]`  (e.g., 1 = B-PER, 2 = I-PER)

If the tokenizer splits "John" into `["Jo", "##hn"]` and "Smith" into `["Smith"]`,  
the tokenized sequence is: `["[CLS]", "Jo", "##hn", "Smith", "[SEP]"]`

**Alignment:**
- `[CLS]` → -100 (special token, ignored)
- `"Jo"` → 1 (B-PER, first subword of "John")
- `"##hn"` → -100 (subword, ignored for loss)
- `"Smith"` → 2 (I-PER)
- `[SEP]` → -100 (special token, ignored)

**Result:**
- Output: `[-100, 1, -100, 2, -100]`

`labels_to_ids` is for converting raw string labels to numbers.

`tokenize_and_align_labels` is for matching those numbers to the actual tokens the model will see, including handling subwords and special tokens.


In [17]:
# Tokenize datasets

tokenized_train = train_dataset.map(tokenize_and_align_labels, batched=True, remove_columns=train_dataset.column_names)
tokenized_dev = dev_dataset.map(tokenize_and_align_labels, batched=True, remove_columns=dev_dataset.column_names)
tokenized_test = test_dataset.map(tokenize_and_align_labels, batched=True, remove_columns=test_dataset.column_names)


Map:   0%|          | 0/12543 [00:00<?, ? examples/s]

Map:   0%|          | 0/2001 [00:00<?, ? examples/s]

Map:   0%|          | 0/2077 [00:00<?, ? examples/s]

In [9]:
# Inspect a few training samples after tokenization

for index in random.sample(range(len(tokenized_train)), 3):
    print(f"Sample {index} of the training set: {tokenized_train[index]}")

Sample 10476 of the training set: {'tokens': ['Overall', ',', 'I', 'was', 'very', 'happy', 'with', 'the', 'customer', 'service', 'and', 'my', 'purchase', '.'], 'ner_tags': [6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6], 'input_ids': [101, 8007, 117, 146, 1108, 1304, 2816, 1114, 1103, 8132, 1555, 1105, 1139, 4779, 119, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'labels': [-100, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, -100]}
Sample 1824 of the training set: {'tokens': ['Once', 'he', 'has', 'properly', 'settled', 'down', ',', 'got', 'use', 'to', 'the', 'new', 'smells', 'and', 'noises', 'he', 'will', 'be', 'fine', '.'], 'ner_tags': [6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6], 'input_ids': [101, 2857, 1119, 1144, 7513, 3035, 1205, 117, 1400, 1329, 1106, 1103, 1207, 16533, 1105, 16256, 1119, 1209, 1129, 2503, 119, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

The values in `input_ids` look random, but they are not! Each number is the unique ID of a token (word or subword) in the BERT model’s vocabulary.

- BERT uses a fixed vocabulary file, where every word, subword, or special token (like `[CLS]`, `[SEP]`) is assigned a unique integer.
- When you tokenize text, each token is replaced by its corresponding ID from this vocabulary.
- For example, `[CLS]` is always 101, `[SEP]` is always 102, and other words/subwords have their own fixed IDs.

So, the numbers are not random—they are consistent and determined by the pretrained model’s vocabulary. If you tokenize the same word with the same tokenizer, you’ll always get the same ID.

## Initialize model and prepare data for loading

In [18]:
# Initialize the model

model = AutoModelForTokenClassification.from_pretrained(model_name, config=config)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: google-bert/bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly init

When you use BERT for a new task (like NER), a new "classification head" (the final layer for your specific labels) is added. This head did not exist in the original pretrained model, so its weights and biases are randomly initialized.
The rest of the model (the BERT layers) loads the pretrained weights.

In [23]:
# Use the processed dataset and data collator to build PyTorch DataLoader objects

# Prepares batches of data for token classification, handling padding and label alignment
data_collator = DataCollatorForTokenClassification(tokenizer)

# Creating DataLoader objects for training and development datasets, needed for pyTorch training loop
train_dataloader = DataLoader(tokenized_train, shuffle=True, collate_fn=data_collator, batch_size=12)
dev_dataloader = DataLoader(tokenized_dev, collate_fn=data_collator, batch_size=12)

In [24]:
# Move model to device (CPU/GPU)
if torch.cuda.is_available():
    device = "cuda"
    print('Moved model to GPU')
else:
    device = "cpu"
    print('Moved model to CPU')

model.to(device)

Moved model to CPU


BertForTokenClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(28996, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, el

On most laptops without a compatible GPU, torch.cuda.is_available() will be False, so device will be set to "cpu".
The model and data will run on your CPU, which is slower than a GPU but still works for smaller experiments and learning

## Training

In [27]:
# Define hyperparameters (now set to assignment's parameters)

learning_rate = 2e-5
num_train_epochs = 5

In [28]:
# Create optimizer (e.g. AdamW)

optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

In [ ]:
# Training loop

model.train()
for epoch in range(num_train_epochs):
    total_loss = 0
    # Showing progress bar with tqdm
    pbar = tqdm(enumerate(train_dataloader), total=len(train_dataloader), desc=f"Training Epoch {epoch+1}")
    for step, batch in pbar:
        # Move batch to device, for hpc
        batch = {k: v.to(device) for k, v in batch.items()}
        # Zero gradients each batch
        optimizer.zero_grad()
        # Forward pass
        outputs = model(**batch) # Predictions
        loss = outputs.loss # Cross entropy loss
        # Backward pass
        loss.backward() # Compute gradients
        # Update parameters
        optimizer.step() # Update model parameters with ADAM 
        # Track loss
        total_loss += loss.item()
        pbar.set_postfix({"loss": loss.item()})
    avg_loss = total_loss / len(train_dataloader)
    print(f"Epoch {epoch+1} average loss: {avg_loss:.4f}")

# Save the trained model and tokenizer in model1 folder (gitignore since it is huge)
model.save_pretrained("model1")
tokenizer.save_pretrained("model1")

Training Epoch 1:   0%|          | 0/1046 [00:00<?, ?it/s]

Epoch 1 average loss: 0.0329


Training Epoch 2:   0%|          | 0/1046 [00:00<?, ?it/s]

Epoch 2 average loss: 0.0145


Training Epoch 3:   0%|          | 0/1046 [00:00<?, ?it/s]

...

Epoch 4 average loss: 0.0053


Training Epoch 5:   0%|          | 0/1046 [00:00<?, ?it/s]

Epoch 5 average loss: 0.0028


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('model1\\tokenizer_config.json', 'model1\\tokenizer.json')

## Evaluation on dev

In [30]:
# Load saved model

model = AutoModelForTokenClassification.from_pretrained("model1")
tokenizer = AutoTokenizer.from_pretrained("model1")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [ ]:
# Metric calculation functions (from assignment 6)

metric = evaluate.load("seqeval")

def get_labels(predictions, references):
    true_predictions = []
    true_labels = []
    # For evaluation data needs to be in numpy format
    # PyTorch only allows conversion to numpy arrays when the tensor is on the CPU, so if used hpc before move to cpu now
    predictions = predictions.cpu().numpy() if hasattr(predictions, 'cpu') else np.array(predictions)
    # Ground truth
    references = references.cpu().numpy() if hasattr(references, 'cpu') else np.array(references)

    for pred, ref in zip(predictions, references):
        # Sentence-level predictions and labels
        pred_labels = []
        true_ref = []
        for p, r in zip(pred, ref):
        # Only consider non-subword tokens (those with label ID not equal to -100)
            if r != -100:
                # Convert prediction and labels back to human readable format using id2label mapping
                pred_labels.append(id2label[p])
                true_ref.append(id2label[r])
        true_predictions.append(pred_labels)
        true_labels.append(true_ref)
    return true_predictions, true_labels

def compute_metrics(preds, refs):
    results = metric.compute(predictions=preds, references=refs)
    return {
        "Precision": results["overall_precision"],
        "Recall": results["overall_recall"],
        "F1": results["overall_f1"],
        "Accuracy": results["overall_accuracy"],
    }


In [ ]:
# Evaluation

model.eval()
validation_progress_bar = tqdm(range(len(dev_dataloader)), desc="Evaluating")
# Storage
all_predictions = []
all_labels = []

for step, batch in enumerate(dev_dataloader):
    batch = {k: v.to(device) for k, v in batch.items()}
    with torch.no_grad(): # No gradient calculation during evaluation, for efficiency
        outputs = model(**batch)
    predictions = outputs.logits.argmax(dim=-1) # Get predicted label IDs by taking the index of the highest logit for each token
    labels = batch["labels"] # Ground truth labels
    predicted_labels, true_labels = get_labels(predictions, labels) # Convert predicted and true label IDs to human-readable labels, while filtering out subword tokens
    all_predictions.extend(predicted_labels)
    all_labels.extend(true_labels)
    validation_progress_bar.update(1)

validation_metrics = compute_metrics(all_predictions, all_labels)
print("Validation metrics:", validation_metrics)

Evaluating:   0%|          | 0/167 [00:00<?, ?it/s]

Validation metrics: {'Precision': 0.8010152284263959, 'Recall': 0.8167701863354038, 'F1': 0.8088159917990775, 'Accuracy': 0.9844924251461291}


## Get predictions from masked test set

In [34]:
# Prediction on test set and writing to file

model.eval()
predicted_labels = []

for tokens in tqdm(test_sentences, desc="Predicting on test set"):

    # Tokenize and align for a single sentence
    inputs = tokenizer(tokens, is_split_into_words=True, return_tensors="pt", truncation=True, max_length=128)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model(**inputs)
    predictions = outputs.logits.argmax(dim=-1).squeeze().cpu().numpy()
    word_ids = inputs['input_ids'].squeeze().cpu().numpy()

    # Align predictions to original tokens
    word_ids_map = inputs['input_ids'].squeeze().tolist()
    token_predictions = []
    word_idx = 0
    for i, token_id in enumerate(inputs['input_ids'].squeeze()):
        if tokenizer.convert_ids_to_tokens(int(token_id)).startswith("##") or token_id in [tokenizer.cls_token_id, tokenizer.sep_token_id]:
            continue
        token_predictions.append(predictions[i])
        word_idx += 1
        if word_idx == len(tokens):
            break

    # Convert label IDs to label names
    pred_labels = [id2label[pred] for pred in token_predictions]
    predicted_labels.append(pred_labels)

# Write predictions to file
output_file = "test_predictions.txt"
with open(output_file, "w", encoding="utf-8") as f:
    for tokens, labels in zip(test_sentences, predicted_labels):
        for i, (token, label) in enumerate(zip(tokens, labels)):
            f.write(f"{i+1}\t{token}\t{label}\n")
        f.write("\n")

print(f"Predictions written to {output_file}")

Predicting on test set:   0%|          | 0/2077 [00:00<?, ?it/s]

Predictions written to test_predictions.txt
